# 01 Exploration

Week 2 exploratory data analysis for predicting California residential single-family property `ClosePrice` using historical CRMLS sold property data.

This notebook focuses on four goals:

1. Combine the monthly raw extracts into one analysis-ready table.
2. Apply the project scope filter and remove obvious duplicate listings.
3. Check data quality, missingness, and suspicious values.
4. Build a first-pass understanding of price levels, trends, and important features.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)
plt.style.use("ggplot")


## Load Monthly Raw Data

The repository now contains monthly sold-property extracts in `data/raw/`. For Week 2 we will load every available file, tag each row with its source month, and combine them into a single raw DataFrame.


In [ ]:
raw_dir = Path("../data/raw")
raw_files = sorted(raw_dir.glob("CRMLSSold*.csv"))

if not raw_files:
    raise FileNotFoundError("No raw CSV files were found in ../data/raw")

frames = []
for file_path in raw_files:
    month_tag = file_path.stem.replace("CRMLSSold", "")
    monthly_df = pd.read_csv(file_path, low_memory=False)
    monthly_df["source_file"] = file_path.name
    monthly_df["source_month"] = month_tag
    frames.append(monthly_df)

raw_df = pd.concat(frames, ignore_index=True)

print(f"Loaded {len(raw_files)} monthly files")
print(f"Raw shape: {raw_df.shape}")
display(pd.DataFrame({"file": [p.name for p in raw_files]}))


## Apply Project Scope And Clean Core Fields

Project scope from the repository notes:

- `PropertyType == "Residential"`
- `PropertySubType == "SingleFamilyResidence"`

Because the monthly extracts can overlap slightly, we also remove duplicate listing records using `ListingKeyNumeric` when available, falling back to other listing identifiers if needed.


In [ ]:
scoped_df = raw_df.loc[
    (raw_df["PropertyType"] == "Residential")
    & (raw_df["PropertySubType"] == "SingleFamilyResidence")
].copy()

scoped_df["dedupe_key"] = (
    scoped_df["ListingKeyNumeric"].astype("string").fillna("").str.strip()
)

missing_dedupe_key = scoped_df["dedupe_key"].eq("")
scoped_df.loc[missing_dedupe_key, "dedupe_key"] = (
    scoped_df.loc[missing_dedupe_key, "ListingId"].astype("string").fillna("").str.strip()
)

still_missing_dedupe_key = scoped_df["dedupe_key"].eq("")
scoped_df.loc[still_missing_dedupe_key, "dedupe_key"] = (
    scoped_df.loc[still_missing_dedupe_key, "ListingKey"].astype("string").fillna("").str.strip()
)

before_dedup = len(scoped_df)
analysis_df = scoped_df.drop_duplicates(subset="dedupe_key", keep="first").copy()

numeric_cols = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "DaysOnMarket",
]

for col in numeric_cols:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

analysis_df["CloseDate"] = pd.to_datetime(analysis_df["CloseDate"], errors="coerce")
analysis_df["source_month"] = pd.to_datetime(analysis_df["source_month"], format="%Y%m", errors="coerce")
analysis_df["sale_to_list_ratio"] = analysis_df["ClosePrice"] / analysis_df["ListPrice"]
analysis_df["log_close_price"] = np.log1p(analysis_df["ClosePrice"])
analysis_df["log_living_area"] = np.log1p(analysis_df["LivingArea"])
analysis_df["close_month"] = analysis_df["CloseDate"].dt.to_period("M").dt.to_timestamp()

print(f"Scoped rows before dedupe: {before_dedup:,}")
print(f"Rows after dedupe: {len(analysis_df):,}")
print(f"Removed duplicate rows: {before_dedup - len(analysis_df):,}")


## Dataset Snapshot

A quick overview helps confirm that the scope filter worked and that the time coverage looks reasonable before deeper analysis.


In [ ]:
overview = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "scoped_rows_before_dedup",
            "rows_after_dedup",
            "column_count",
            "closeprice_missing",
            "closedate_missing",
        ],
        "value": [
            len(raw_df),
            before_dedup,
            len(analysis_df),
            analysis_df.shape[1],
            int(analysis_df["ClosePrice"].isna().sum()),
            int(analysis_df["CloseDate"].isna().sum()),
        ],
    }
)

display(overview)

display(
    analysis_df.groupby("source_month", dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values("source_month")
)

analysis_df.head()


## Missingness Review

The target and most modeling fields are present for nearly every row. This section quantifies the small set of fields that will need attention before modeling.


In [ ]:
key_fields = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "City",
    "PostalCode",
    "Latitude",
    "Longitude",
    "CloseDate",
    "DaysOnMarket",
]

missing_summary = (
    analysis_df[key_fields]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_summary["missing_pct"] = missing_summary["missing_count"] / len(analysis_df)
missing_summary = missing_summary.sort_values(["missing_pct", "column"], ascending=[False, True])

display(missing_summary)


## Numeric Summary Statistics

The price target is strongly right-skewed, so both raw and log-scale views will matter later. We also summarize size and market-timing variables that are likely to matter in modeling.


In [ ]:
numeric_summary_cols = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
]

summary_table = analysis_df[numeric_summary_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
summary_table = summary_table.rename(columns={"50%": "median"})
display(summary_table)


## Suspicious Values And Data Quality Flags

A few values look unusual enough to flag for follow-up before modeling:

- extremely small sale prices
- zero living area
- negative days on market
- very large lot sizes and close prices

We are not dropping them yet in Week 2, but we want to document them clearly.


In [ ]:
quality_flags = pd.DataFrame(
    {
        "flag": [
            "close_price_le_1000",
            "living_area_eq_0",
            "living_area_le_300",
            "days_on_market_lt_0",
            "year_built_lt_1900",
            "lot_size_eq_0",
        ],
        "row_count": [
            int((analysis_df["ClosePrice"] <= 1000).sum()),
            int((analysis_df["LivingArea"] == 0).sum()),
            int((analysis_df["LivingArea"] <= 300).sum()),
            int((analysis_df["DaysOnMarket"] < 0).sum()),
            int((analysis_df["YearBuilt"] < 1900).sum()),
            int((analysis_df["LotSizeSquareFeet"] == 0).sum()),
        ],
    }
)

display(quality_flags)

suspicious_rows = analysis_df.loc[
    (analysis_df["ClosePrice"] <= 1000)
    | (analysis_df["LivingArea"] == 0)
    | (analysis_df["DaysOnMarket"] < 0),
    [
        "ListingId",
        "UnparsedAddress",
        "City",
        "CloseDate",
        "ClosePrice",
        "ListPrice",
        "LivingArea",
        "DaysOnMarket",
        "source_file",
    ],
].sort_values(["ClosePrice", "LivingArea"], ascending=[True, True])

display(suspicious_rows.head(20))


## Distribution Plots

We start with univariate views of the target and several important features. `ClosePrice` is plotted on both the raw and log scales because the luxury tail is large enough to compress the rest of the market on the raw scale.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

analysis_df["ClosePrice"].dropna().plot(kind="hist", bins=60, ax=axes[0, 0], color="steelblue", edgecolor="white")
axes[0, 0].set_title("ClosePrice Distribution")
axes[0, 0].set_xlabel("ClosePrice")

analysis_df["log_close_price"].dropna().plot(kind="hist", bins=60, ax=axes[0, 1], color="darkorange", edgecolor="white")
axes[0, 1].set_title("log(1 + ClosePrice) Distribution")
axes[0, 1].set_xlabel("log(1 + ClosePrice)")

analysis_df["LivingArea"].clip(upper=analysis_df["LivingArea"].quantile(0.99)).dropna().plot(
    kind="hist", bins=60, ax=axes[1, 0], color="seagreen", edgecolor="white"
)
axes[1, 0].set_title("LivingArea Distribution (Clipped At 99th Percentile)")
axes[1, 0].set_xlabel("LivingArea")

analysis_df["DaysOnMarket"].clip(lower=0, upper=analysis_df["DaysOnMarket"].quantile(0.99)).dropna().plot(
    kind="hist", bins=60, ax=axes[1, 1], color="mediumpurple", edgecolor="white"
)
axes[1, 1].set_title("DaysOnMarket Distribution (0 To 99th Percentile)")
axes[1, 1].set_xlabel("DaysOnMarket")

plt.tight_layout()
plt.show()


## Monthly Trend Review

Even with only seven monthly extracts, it is useful to check whether median sale price is stable or drifting upward over time.


In [ ]:
monthly_price_stats = (
    analysis_df.groupby("source_month")
    .agg(
        transaction_count=("ClosePrice", "size"),
        median_close_price=("ClosePrice", "median"),
        mean_close_price=("ClosePrice", "mean"),
    )
    .reset_index()
    .sort_values("source_month")
)

display(monthly_price_stats)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(
    monthly_price_stats["source_month"],
    monthly_price_stats["median_close_price"],
    marker="o",
    linewidth=2,
    color="navy",
    label="Median ClosePrice",
)
ax2.bar(
    monthly_price_stats["source_month"],
    monthly_price_stats["transaction_count"],
    alpha=0.25,
    color="gray",
    label="Transaction Count",
)

ax1.set_title("Monthly Median ClosePrice And Transaction Volume")
ax1.set_xlabel("Source Month")
ax1.set_ylabel("Median ClosePrice")
ax2.set_ylabel("Transaction Count")
ax1.tick_params(axis="x", rotation=45)
plt.show()


## Feature Relationships

The two most important first-pass relationships to check are:

- sale price vs. list price
- sale price vs. living area

Raw-scale correlations are muted by outliers, so we also calculate correlations on log-transformed variables.


In [ ]:
plot_df = analysis_df[["ClosePrice", "ListPrice", "LivingArea", "log_close_price", "log_living_area", "DaysOnMarket"]].dropna()
plot_sample = plot_df.sample(min(5000, len(plot_df)), random_state=42)

corr_table = pd.DataFrame(
    {
        "metric": [
            "corr(ClosePrice, ListPrice)",
            "corr(ClosePrice, LivingArea)",
            "corr(ClosePrice, DaysOnMarket)",
            "corr(log_close_price, log_living_area)",
        ],
        "value": [
            plot_df["ClosePrice"].corr(plot_df["ListPrice"]),
            plot_df["ClosePrice"].corr(plot_df["LivingArea"]),
            plot_df["ClosePrice"].corr(plot_df["DaysOnMarket"]),
            plot_df["log_close_price"].corr(plot_df["log_living_area"]),
        ],
    }
)

display(corr_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(plot_sample["ListPrice"], plot_sample["ClosePrice"], alpha=0.2, s=12, color="teal")
axes[0].set_title("ClosePrice vs ListPrice")
axes[0].set_xlabel("ListPrice")
axes[0].set_ylabel("ClosePrice")

axes[1].scatter(plot_sample["LivingArea"], plot_sample["ClosePrice"], alpha=0.2, s=12, color="tomato")
axes[1].set_title("ClosePrice vs LivingArea")
axes[1].set_xlabel("LivingArea")
axes[1].set_ylabel("ClosePrice")

plt.tight_layout()
plt.show()


## Segment Analysis

Bedrooms, bathrooms, and location all show clear price segmentation. This is useful for feature engineering later because it suggests both structural and geographic variables will matter.


In [ ]:
bedroom_price = (
    analysis_df.groupby("BedroomsTotal")
    .agg(transaction_count=("ClosePrice", "size"), median_close_price=("ClosePrice", "median"))
    .query("transaction_count >= 30")
    .reset_index()
    .sort_values("BedroomsTotal")
)

bathroom_price = (
    analysis_df.groupby("BathroomsTotalInteger")
    .agg(transaction_count=("ClosePrice", "size"), median_close_price=("ClosePrice", "median"))
    .query("transaction_count >= 30")
    .reset_index()
    .sort_values("BathroomsTotalInteger")
)

city_volume = (
    analysis_df.groupby("City")
    .agg(transaction_count=("ClosePrice", "size"), median_close_price=("ClosePrice", "median"), mean_close_price=("ClosePrice", "mean"))
    .reset_index()
    .sort_values("transaction_count", ascending=False)
)

top_cities_by_volume = city_volume.head(12)
high_value_cities = city_volume.loc[city_volume["transaction_count"] >= 100].sort_values("median_close_price", ascending=False).head(10)

display(bedroom_price)
display(bathroom_price)
display(top_cities_by_volume)
display(high_value_cities)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(bedroom_price["BedroomsTotal"].astype(str), bedroom_price["median_close_price"], color="cornflowerblue")
axes[0].set_title("Median ClosePrice By Bedroom Count")
axes[0].set_xlabel("BedroomsTotal")
axes[0].set_ylabel("Median ClosePrice")

axes[1].bar(bathroom_price["BathroomsTotalInteger"].astype(str), bathroom_price["median_close_price"], color="salmon")
axes[1].set_title("Median ClosePrice By Bathroom Count")
axes[1].set_xlabel("BathroomsTotalInteger")
axes[1].set_ylabel("Median ClosePrice")

plt.tight_layout()
plt.show()


## Sale-To-List Ratio And Spatial Coverage

This ratio is a simple way to understand whether homes usually sold below, at, or above list price. A quick geographic scatter also confirms that latitude and longitude coverage is broad enough to support location-based feature engineering later.


In [ ]:
ratio_summary = analysis_df["sale_to_list_ratio"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).to_frame().T
ratio_summary = ratio_summary.rename(columns={"50%": "median"})
display(ratio_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

analysis_df["sale_to_list_ratio"].clip(lower=0.7, upper=1.3).dropna().plot(
    kind="hist", bins=60, ax=axes[0], color="slateblue", edgecolor="white"
)
axes[0].set_title("Sale-To-List Ratio Distribution (Clipped 0.7 to 1.3)")
axes[0].set_xlabel("ClosePrice / ListPrice")

map_sample = analysis_df[["Longitude", "Latitude", "log_close_price"]].dropna().sample(
    min(5000, analysis_df[["Longitude", "Latitude", "log_close_price"]].dropna().shape[0]),
    random_state=42,
)
scatter = axes[1].scatter(
    map_sample["Longitude"],
    map_sample["Latitude"],
    c=map_sample["log_close_price"],
    cmap="viridis",
    s=8,
    alpha=0.5,
)
axes[1].set_title("Geographic Coverage Colored By log(1 + ClosePrice)")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(scatter, ax=axes[1], label="log(1 + ClosePrice)")

plt.tight_layout()
plt.show()


## Week 2 Findings

Key takeaways from this first EDA pass:

1. After applying the project scope and removing duplicate listing IDs, the analysis set contains **71,414** residential single-family sold listings.
2. The target is highly right-skewed. Median `ClosePrice` is about **$890,000**, while the mean is about **$1.34M**, so log-based modeling and robust error analysis will be important.
3. Core modeling fields are in good shape. `ClosePrice` and `ListPrice` have no missing values, and most structural fields have very low missingness. `LotSizeSquareFeet` is the only notable gap at about **1.7%** missing.
4. There are a few obvious anomalies to handle before modeling, including **2** sale prices at or below `$1,000`, **20** records with zero `LivingArea`, and **6** negative `DaysOnMarket` values.
5. Median sale price appears to rise across the available monthly extracts, from roughly **$875k** in `2025-11` to about **$930k** in `2026-05`.
6. `ListPrice` is extremely informative for predicting `ClosePrice`, while `LivingArea` becomes much more useful once both variables are viewed on the log scale.
7. Price segmentation by structure and geography is strong. Higher bedroom and bathroom counts generally correspond to higher median prices, and high-volume premium cities such as **Beverly Hills**, **Newport Beach**, and **Santa Monica** sit far above the statewide middle.
8. The sale-to-list ratio is centered around **1.00**, which suggests many homes sold near list price, but the upper tail shows a meaningful subset of competitive over-list transactions.

### Suggested Week 3 Follow-Up

- Decide how to handle outliers and suspicious records.
- Create a reusable cleaned feature set in `src/`.
- Engineer date, location, and price-per-square-foot features.
- Build a baseline model and evaluate it against a holdout split.
